# HuggingFace Transformers Text Classification with Progression Tracking

This example demonstrates how to fine-tune a transformer model for text classification using the [IMDB movie review dataset](https://huggingface.co/datasets/imdb) and [HuggingFace Transformers](https://huggingface.co/docs/transformers).

This notebook walks you through:
- Running training on Kubernetes with automatic progression tracking
- Mounting persistent storage (RWX PVC) for efficient model/dataset caching
- Using rank 0 to download once, then all workers load from shared storage
- Scaling across multiple nodes with distributed training
- Monitoring training progress in real-time without any extra code

**Key Feature**: TransformersTrainer automatically tracks training progress (steps, epochs, metrics) without requiring any modifications to your training code!


## Install the Kubeflow SDK

Install the Kubeflow SDK interact with Kubeflow Trainer APIs:

In [ ]:
%pip install --force-reinstall ../dist/kubeflow-0.2.0-py3-none-any.whl

## Define the Training Function

Create a function to fine-tune DistilBERT on the IMDB sentiment classification task.

**Note**: This is standard HuggingFace Transformers code with NO special instrumentation needed for progression tracking!


In [7]:
def train_sentiment_classifier_sft():
    """SFT-based distributed training - 1 process per CPU."""
    import os
    import torch.distributed as dist
    from datasets import load_dataset, load_from_disk
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer, SFTConfig

    # Initialize distributed process group FIRST
    dist.init_process_group(backend="gloo")  # Use gloo for CPU
    
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    
    print(f"[Rank {rank}/{world_size}] Starting (local_rank={local_rank})")
    
    model_path = "/tmp/workspace/model"
    dataset_path = "/tmp/workspace/dataset"
    
    # Only rank 0 downloads and creates directories
    if rank == 0:
        print(f"[Rank {rank}] Creating directories and downloading...")
        os.makedirs(model_path, exist_ok=True)
        os.makedirs(dataset_path, exist_ok=True)
        
        model = AutoModelForCausalLM.from_pretrained("gpt2")
        tokenizer = AutoTokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
        
        model.save_pretrained(model_path)
        tokenizer.save_pretrained(model_path)
        
        dataset = load_dataset("stanfordnlp/imdb")
        dataset.save_to_disk(dataset_path)
        print(f"[Rank {rank}] Save complete - model and data ready")
    
    # Wait for rank 0 to finish saving
    print(f"[Rank {rank}] Waiting at barrier...")
    dist.barrier()
    print(f"[Rank {rank}] Passed barrier")
    
    # All ranks load
    print(f"[Rank {rank}] Loading model from {model_path}")
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tokenizer.pad_token = tokenizer.eos_token
    
    print(f"[Rank {rank}] Loading dataset from {dataset_path}")
    dataset = load_from_disk(dataset_path)
    
    train_dataset = dataset["train"].select(range(1000))
    eval_dataset = dataset["test"].select(range(200))
    
    print(f"[Rank {rank}] Dataset ready: {len(train_dataset)} train samples")
    
    def formatting_func(example):
        sentiment = "positive" if example["label"] == 1 else "negative"
        return f"Review: {example['text'][:200]}\nSentiment: {sentiment}"
    
    sft_config = SFTConfig(
        output_dir="./results",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_steps=50,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=False,
        bf16=False,
        use_cpu=True,
        ddp_find_unused_parameters=False,
        max_grad_norm=1.0,
        report_to=[],
        max_seq_length=256,
        dataset_text_field="text",
        packing=False,
    )
    
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        formatting_func=formatting_func,
        processing_class=tokenizer,
    )
    
    print(f"[Rank {rank}] Starting training...")
    trainer.train()
    
    print(f"[Rank {rank}] Complete!")

## Run Training with PVC (Rank 0 Downloads)

This approach mounts a **ReadWriteMany (RWX) PVC** to `/workspace`:

**How it works**:
1. **PVC mounted at `/workspace`** - shared storage across all worker pods
2. **Rank 0 downloads** model and dataset to `/workspace/model` and `/workspace/dataset`
3. **Other ranks wait** (via `dist.barrier()`) then load from shared storage
4. **Progression tracking** automatically enabled (tracks steps, epochs, metrics)

**To use**: Replace `"rwx-pvc-name"` with your actual PVC name. On subsequent runs, rank 0 will skip downloading if files exist.


In [ ]:
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.options import ContainerOverride, PodSpecOverride, PodTemplateOverride, PodTemplateOverrides

# Initialize Kubernetes backend (default)
from kubeflow.trainer.backends.kubernetes.backend import KubernetesBackendConfig
from kubernetes import client

api_server = "https://api.abdhumal-osd.qic7.p1.openshiftapps.com:6443"
token = "sha256~E_EAbhc-nJWnTEqATY3U05PH3CLlYB0xNdjl7Dx_HV4"

configuration = client.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}

# Un-comment if your cluster API server uses a self-signed certificate or an un-trusted CA
configuration.verify_ssl = True

api_client = client.ApiClient(configuration)
trainer_client = TrainerClient(backend_config= KubernetesBackendConfig(client_configuration=api_client.configuration,namespace="abdhumal-test"))

print("Available runtimes :", len(trainer_client.list_runtimes()))
for r in trainer_client.list_runtimes():
    print(f"- {r.name}")


In [13]:
# Submit training job with TransformersTrainer
# Progression tracking is ENABLED BY DEFAULT (no extra code needed!)
from kubeflow.trainer.options import Name, Labels
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig

job_name = trainer_client.train(
    trainer=TransformersTrainer(
        func=train_sentiment_classifier_sft,
        packages_to_install=[],
        num_nodes=2,  # ← Scale to 2 nodes
        resources_per_node={
            "cpu": 2,
            "memory": "14Gi",
            # "nvidia.com/gpu": 1,  # 1 GPU per node = 2 GPUs total
        },
        # enable training progression tracking
        enable_progression_tracking=True,  # This is the default!
        metrics_port=28080,  # Default port for metrics server
        metrics_poll_interval_seconds=10,  # Default: 30 seconds, How often to poll metrics
        
        #enable JIT checkpointing
        enable_jit_checkpoint=True,
        output_dir="pvc://my-pvc/checkpoints",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="epoch",
            save_total_limit=3
        )
    ),
    runtime=trainer_client.get_runtime("torch-cuda-251"),
    # Mount RWX PVC at /workspace - rank 0 downloads model/dataset, other ranks load from it
    options=[
        Name(name="trl-job"),
        Labels(labels={"kueue.x-k8s.io/queue-name": "test-lq"}),
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    volumes=[
                        {"name": "workspace", "persistentVolumeClaim": {"claimName": "rwx-pvc-name"}}
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {"name": "workspace", "mountPath": "/tmp/workspace"}  # Mount under /tmp
                            ]
                        )
                    ]
                )
            )
        )
    ]
)

/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [15]:
# check job status
job = trainer_client.get_job("trl-job")
print(f"Final TrainJob Status:")
print(f"   Name: {job.name}")
print(f"   Status: {job.status}")
print(f"   Created: {job.creation_timestamp}")
print(f"   Nodes: {job.num_nodes}")
print(f"   Runtime: {job.runtime.name}")

if job.steps:
    print(f"   Steps:")
    for step in job.steps:
        print(f"     - {step.name}: {step.status}")
    print()

Final TrainJob Status:
   Name: trl-job
   Status: Complete
   Created: 2025-11-24 15:50:23+00:00
   Nodes: 2
   Runtime: torch-cuda-251
   Steps:
     - node-0: Succeeded
     - node-1: Succeeded



## Progression Tracking

TransformersTrainer automatically tracks training progress (steps, epochs, loss, metrics) via HTTP endpoint on port 28080. No code changes needed!

### Fetch Metrics from Rank 0 Pod

Query the metrics endpoint directly from the rank 0 training pod:


In [25]:
from kubernetes import client as k8s_client, config
import json

config.load_kube_config()
custom_api = k8s_client.CustomObjectsApi()

trainjob_name = "trl-job"
job_namespace = "abdhumal-test"

try:
    trainjob = custom_api.get_namespaced_custom_object(
        group="trainer.kubeflow.org",
        version="v1alpha1",
        namespace=job_namespace,
        plural="trainjobs",
        name=trainjob_name
    )
    
    annotations = trainjob.get('metadata', {}).get('annotations', {})
    
    print(f"TrainJob Annotations for {trainjob_name}:\n")
    
    # Progression tracking config
    print("Config:")
    print(f"  tracking-enabled: {annotations.get('trainer.opendatahub.io/progression-tracking', 'N/A')}")
    print(f"  metrics-port: {annotations.get('trainer.opendatahub.io/metrics-port', 'N/A')}")
    print(f"  poll-interval: {annotations.get('trainer.opendatahub.io/metrics-poll-interval', 'N/A')}s")
    
    # Progression metrics (if controller populated them)
    if annotations.get('trainer.opendatahub.io/progression-tracking', 'N/A') == "true":
        trainerStatus=annotations.get('trainer.opendatahub.io/trainerStatus', 'N/A')
        metrics=json.loads(trainerStatus)
        print("\nMetrics:")
        print(f"  progress: { metrics["progressPercentage"]}%")
        print(f"  step: { metrics["currentStep"]}/{metrics["totalSteps"]}")
        print(f"  epoch: {metrics["currentEpoch"]}/{metrics["totalEpochs"]}")
        print(f"  remaining-time : {metrics["estimatedRemainingSeconds"]}s")
    print(f"\nAll annotations:\n{json.dumps(annotations, indent=2)}")
    
except Exception as e:
    print(f"Error: {e}")

TrainJob Annotations for trl-job:

Config:
  tracking-enabled: true
  metrics-port: 28080
  poll-interval: 10s

Metrics:
  progress: 100%
  step: 124/124
  epoch: 1.976/2
  remaining-time : 0s

All annotations:
{
  "kueue.x-k8s.io/trainjob-override-idx": "0",
  "trainer.opendatahub.io/metrics-poll-interval": "10",
  "trainer.opendatahub.io/metrics-port": "28080",
  "trainer.opendatahub.io/progression-tracking": "true",
  "trainer.opendatahub.io/trainerStatus": "{\"progressPercentage\":100,\"estimatedRemainingSeconds\":0,\"currentStep\":124,\"totalSteps\":124,\"currentEpoch\":1.976,\"totalEpochs\":2,\"trainMetrics\":{\"grad_norm\":7.2895684242248535,\"learning_rate\":0.0000013513513513513515,\"loss\":3.416},\"evalMetrics\":{\"eval_loss\":3.4455575942993164,\"eval_mean_token_accuracy\":0.3912548583287459,\"eval_num_tokens\":107052,\"eval_runtime\":7.4837,\"eval_samples_per_second\":26.725,\"eval_steps_per_second\":1.737},\"lastUpdatedTime\":\"2025-11-24T15:58:37Z\"}"
}


In [26]:
trainer_client.delete_job(job_name)

/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
